# Training and Test Errors, Cross-Validation, Pipelines

In [1]:
import pandas as pd
import numpy as np

**Use scikit-learn, including pipelines, whenever possible.**

## Ames Housing Data


We have fit different regression models to predict house prices (SalePrice, not log). Now we will evaluate models by estimating their prediction error.

In [2]:
df_ames = pd.read_csv('AmesHousing.txt', sep='\t')
df_ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


Since all of the houses in the data set have a SalePrice, we'll include all the observations in the training data. We'll start with a basic linear regression model with a few features.

In [3]:
X_train = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath"]]

y_train = df_ames["SalePrice"]

In [4]:
from sklearn.linear_model import LinearRegression

model1 = LinearRegression()
model1.fit(
    X=X_train,
    y=y_train
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


1. Explain conceptually what the "fit" step does. What is the output?

Finding the best intercept and coefficients that minimizes distance between the observed response value and the predicted/expected value for the response

In [5]:
round(model1.intercept_)
pd.Series(model1.coef_, index = X_train.columns)

Gr Liv Area        118.099868
Bedroom AbvGr   -29994.956760
Full Bath        26728.533428
Half Bath         1271.138477
dtype: float64

Equation:
$$\hat{y}_i = 46942 + 118.10\times x_{1i} - 2999496\times x_{2i} + 26728.53\times x_{3i} + 1271.14\times x_{4i}$$

2. Explain conceptually how you would compute the *training* RMSE (root mean square error) of this model.

How far the predicted values are from the observed values in the same units as the observed values

In [6]:
predicted = model1.predict(X_train)
residual_squared = (y_train - predicted) **2
by_hand_rmse = np.sqrt(residual_squared.mean())


3. Interpret the *training RMSE*. Is it big or small? What could you use as a rough benchmark for comparison?

RMSE is how far each observed value deviates from the predicted value on average, and the mechanism to determine whether that value is big or small is to compare the RMSE to the standard deviation of the response variable.

So if the RMSE is smaller than the standard deviation of the response variable then the model has explained the variability in the response variable.

In [7]:
y_train.std()

79886.69235666493

4. Suppose we add more features like "TotRms AbvGrd" and fit a new model. Would the *training* RMSE increase or decrease? Why is comparing the *training* RMSE of these two models not useful?

When we add more features into the model, the RMSE would either stay the same or decrease. When comparing these two models, just using RMSE is not useful becasue by statistical modeling the RMSE for the more complicated variable will either be the same or smaller than the less complicated model.

5. If we fit a 1-nearest neighbor model, what should the *training* RMSE be?

It should be 0 in that when we choose the nearest neighbor from the training dataset, we should essentially pick the same value as the predicted value. 

However, there are times when there can be ties that would cause the RMSE not to be exactly 0.

6. What happens to the *training* RMSE of a $k$-nearest neighbors model as $k$ increases?

As k increases, the RMSE gets closer to the standard deviation

In [8]:
from sklearn.neighbors import KNeighborsRegressor

rows = []
for k in [1, 200, 500, 1000, len(X_train)]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train, y_train)
    predicted = knn.predict(X_train)

    residual_squared = (y_train - predicted) ** 2
    rmse = np.sqrt(residual_squared.mean())
    rows.append([k, rmse])

pd.DataFrame(rows, columns = ["k", "training RMSE"])

,k,training RMSE
0,1,21903.066672
1,200,56257.636176
2,500,58071.482314
3,1000,61025.797875
4,2930,79873.058652


7. We are really interested in the *test* RMSE, not the *training* RMSE. Explain why in practice you can not compute the *test* RMSE.

We are going to use the RMSE of the test data to compare models rather than compare the RMSE of the training data. In practice we may not have access to the target variable of the test data. So we have to use the training information to determine which model to use.

8. Explain conceptually how cross-validations *estimates* the *test* RMSE.

State that K-fold Cross Validation, break up the data into K folds and run a model and calculate the RMSE for each fold. K=10:

1. Cut the data set into 10 folds
2. Drop 1 of the 10 folds, use the other 9 folds to create a model and test it on the dropped fold
3. Repeat this process for each fold
4. Average of the RMSE to the RMSE for the test data set

9. We'll use cross-validation to estimate the *test* RMSE for the model from part 1. Will the *test* RMSE be greater than, less than, or equal to the *training* RMSE for this model?

I expect the cross-validation RMSE to be somewhat higher because each fold contains houses the fitted model did not see.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=10, shuffle=True, random_state=5544)
base_features = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath"]
base_fold_rmse = -cross_val_score(
    LinearRegression(), df_ames[base_features], y_train,
    scoring="neg_root_mean_squared_error", cv=cv
)
print(f"Training RMSE: ${by_hand_rmse:,.0f}")
print(f"Mean 10-fold CV RMSE: ${base_fold_rmse.mean():,.0f}")

10. Suppose we add features like we did in part 4. Will the *test* RMSE necessarily decrease? Fit the test RMSE and compare the two models - which is better based on this criteria?

No. An extra predictor always gives ordinary least squares at least as much flexibility on the training data, but it can make predictions on held-out data worse. Here I add `TotRms AbvGrd` and compare both models on the same ten folds. The original model has the lower mean CV RMSE ($51,354 versus $51,374), so it wins this particular comparison.


In [10]:
expanded_features = base_features + ["TotRms AbvGrd"]
expanded_fold_rmse = -cross_val_score(
    LinearRegression(), df_ames[expanded_features], y_train,
    scoring="neg_root_mean_squared_error", cv=cv
)
print(f"Original four predictors: ${base_fold_rmse.mean():,.0f}")
print(f"With TotRms AbvGrd:     ${expanded_fold_rmse.mean():,.0f}")
print("Lower CV RMSE:", "expanded model" if expanded_fold_rmse.mean() < base_fold_rmse.mean() else "original model")

Original four predictors: $51,354
With TotRms AbvGrd:     $51,374
Lower CV RMSE: original model


11. Now suppose we add a categorical predictor like Neighborhood. The crossval code below will (probably) error. Explain why. Hint: think about how the cross-validation process works and how categorical variables are treated.

Each fold refits the encoder on only nine-tenths of the data. If a neighborhood occurs in the held-out fold but not in that fold's training rows, the default `OneHotEncoder` raises an unknown-category error during prediction. `handle_unknown="ignore"` maps such a category to zeros for that categorical block. The preprocessing must stay inside the pipeline so it is refit separately on each training fold.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

ct = ColumnTransformer(
    [("standardize", StandardScaler(), ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath"]),
     ("dummify", OneHotEncoder(handle_unknown="ignore"), ["Neighborhood"])],
    remainder="drop",
)

neighborhood_model = Pipeline([
    ("preprocessing", ct),
    ("lm", LinearRegression()),
])

X_neighborhood = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood"]]
y_train = df_ames["SalePrice"]
neighborhood_model.fit(X_neighborhood, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('lm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standardize', ...), ('dummify', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [12]:
neighborhood_scores = cross_val_score(
    neighborhood_model, X_neighborhood, y_train,
    scoring="neg_root_mean_squared_error", cv=cv
)
print(f"Neighborhood model mean CV RMSE: ${-neighborhood_scores.mean():,.0f}")

Neighborhood model mean CV RMSE: $39,567


12. Going back to part 1, suppose we had standardized the features first. Would the coefficients of the fitted model change? Would the *training* RMSE change?

Yes, the numerical coefficients change because a one-unit change now means one standard deviation of the original feature. The intercept also changes to reflect the centered predictors. For unregularized linear regression, scaling does not change the fitted predictions (apart from tiny rounding differences), so the training RMSE stays the same.

13. Suppose we want to standardize predictors like in the previous part and estimate test RMSE using cross-validation. Explain how standardization fits in in the CV process. Also explain why it is important to put standardization as part of the model pipeline.

In each fold, fit the scaler's means and standard deviations using that fold's training rows; then transform both its training and validation rows with those values. A pipeline makes `cross_val_score` do this automatically. Standardizing the whole dataset before splitting would let information from validation rows influence the training transformation, causing data leakage and an overly optimistic error estimate.

## Practice Activity

We'll continue with the housing data and our goal of predicting Sale Price.

We will have a competition to see who can come up with the model with the smallest *test* RMSE. Fit several models and try to find the smallest *test* RMSE. Use cross-validation to estimate the *test* RMSE. When you think you have a good model, share your it and its test RMSE with a partner. Then see what other people have done; can you do even better?

Note: next week we will see how to make some of these choices in a more automated way. For now just work by trial and error. You should try:

- Exploring the data first! Make some plots and summaries to get an idea of what might be good predictors to try
- Several different sets of predictors, including both categorical and numerical predictors
- Linear regression models, including models with polynomial features and interactions
- K-nearest neighbors models, inclduing different values of k
- Decision trees, including different sizes of trees

Use pipelines as much as possible, and don't forget to transform variables as needed.

In [25]:
features = ["Gr Liv Area", "Overall Qual", "Full Bath", "TotRms AbvGrd"]
X = df_ames[features]
y = df_ames["SalePrice"]

cv = KFold(n_splits=10, shuffle=True, random_state=5544)

models = {
    "Linear regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "5-neighbor KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ])
}

for name, model in models.items():
    scores = cross_val_score(
        model, X, y, cv=cv,
        scoring="neg_root_mean_squared_error"
    )
    print(name, "average RMSE:", round(-scores.mean()))

Linear regression average RMSE: 41093
5-neighbor KNN average RMSE: 36534


In [26]:
# Refit the same four-predictor KNN pipeline that won the comparison above.
best_model = models["5-neighbor KNN"]
best_model.fit(X, y)
print("Selected model: 5-neighbor KNN, using", features)

Selected model: 5-neighbor KNN, using ['Gr Liv Area', 'Overall Qual', 'Full Bath', 'TotRms AbvGrd']


Among the **two models tested with these four predictors**, five-neighbor KNN had the lower mean ten-fold cross-validation RMSE: **$36,534**, compared with **$41,093** for linear regression. The cell above refits that same KNN pipeline on all 2,930 houses. These are cross-validation estimates; they are not errors measured on a separate test set.